# Lecture 4 Colab: LSTM Time Series Prediction

This notebook demonstrates a simple LSTM time series workflow using a synthetic financial-style series. It is not a trading system.


## Learning objectives

- Generate a synthetic financial-style time series.
- Understand chronological train/test split.
- Convert a time series into sliding windows.
- Train a simple LSTM model.
- Compare LSTM predictions with a naive baseline.
- Understand why smooth predictions do not guarantee good forecasting.


## Connection to Lecture 4

Supports LSTM, time series analysis, time series validation, and Homework 4.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense
from tensorflow.keras.callbacks import EarlyStopping


In [ ]:
np.random.seed(42)
tf.random.set_seed(42)


## Dataset explanation

We generate a synthetic daily transaction volume index with trend, seasonality, noise, and occasional shocks. It is for learning the workflow, not financial prediction.


In [ ]:
n = 600
time = np.arange(n)
noise = np.random.normal(0, 1.8, n)
seasonality = 3 * np.sin(2 * np.pi * time / 30)
trend = 0.03 * time
shocks = np.zeros(n)
shock_days = np.random.choice(np.arange(50, n - 50), size=10, replace=False)
shocks[shock_days] = np.random.normal(0, 8, len(shock_days))
value = 100 + trend + seasonality + noise + shocks
df = pd.DataFrame({"day": time, "value": value})
df.head()


## Plot the full time series

Look for trend, seasonality, noise, and shocks.


In [ ]:
plt.figure(figsize=(10, 4))
plt.plot(df["day"], df["value"], label="Synthetic activity index")
plt.xlabel("Day")
plt.ylabel("Value")
plt.title("Synthetic financial-style time series")
plt.legend()
plt.grid(alpha=0.3)
plt.show()


## Chronological split

For forecasting, training data should come before test data. Random split would leak future information.


In [ ]:
split_idx = int(len(df) * 0.8)
train_df = df.iloc[:split_idx].copy()
test_df = df.iloc[split_idx:].copy()
plt.figure(figsize=(10, 4))
plt.plot(train_df["day"], train_df["value"], label="Train")
plt.plot(test_df["day"], test_df["value"], label="Test")
plt.xlabel("Day")
plt.ylabel("Value")
plt.title("Chronological train/test split")
plt.legend()
plt.grid(alpha=0.3)
plt.show()


## Scaling without leakage

Fit the scaler on training values only. Fitting on all data would use future information.


In [ ]:
scaler = MinMaxScaler()
train_scaled = scaler.fit_transform(train_df[["value"]])
test_scaled = scaler.transform(test_df[["value"]])


## Sliding windows and input shape

Each sample uses the past `window_size` values to predict the next value. LSTM input shape is samples, time steps, features.


In [ ]:
def create_windows(series, window_size):
    X, y = [], []
    for i in range(len(series) - window_size):
        X.append(series[i:i + window_size])
        y.append(series[i + window_size])
    return np.array(X), np.array(y)
window_size = 30
X_train, y_train = create_windows(train_scaled, window_size)
X_test, y_test = create_windows(test_scaled, window_size)
print("X_train shape:", X_train.shape)
print("y_train shape:", y_train.shape)
print("X_test shape:", X_test.shape)
print("y_test shape:", y_test.shape)


## Naive baseline

The naive baseline predicts the previous value as the next value.


In [ ]:
naive_scaled = X_test[:, -1, 0].reshape(-1, 1)
y_test_actual = scaler.inverse_transform(y_test.reshape(-1, 1)).ravel()
naive_pred = scaler.inverse_transform(naive_scaled).ravel()
naive_mae = mean_absolute_error(y_test_actual, naive_pred)
naive_rmse = mean_squared_error(y_test_actual, naive_pred) ** 0.5
print("Naive MAE:", round(naive_mae, 3))
print("Naive RMSE:", round(naive_rmse, 3))


## Build and train a small LSTM

Keep the model small so it runs quickly in Colab.


In [ ]:
model = Sequential([LSTM(32, input_shape=(window_size, 1)), Dense(1)])
model.compile(optimizer="adam", loss="mse")
early_stop = EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True)
model.summary()


In [ ]:
history = model.fit(X_train, y_train, validation_split=0.2, epochs=30, batch_size=32, verbose=1, shuffle=False, callbacks=[early_stop])


## Plot training loss

If validation loss increases while training loss decreases, the model may be overfitting.


In [ ]:
plt.figure(figsize=(7, 4))
plt.plot(history.history["loss"], label="Training loss")
plt.plot(history.history["val_loss"], label="Validation loss")
plt.xlabel("Epoch")
plt.ylabel("MSE loss")
plt.title("Training and validation loss")
plt.legend()
plt.grid(alpha=0.3)
plt.show()


## Predict and inverse transform

Calculate MAE and RMSE on the original value scale.


In [ ]:
lstm_scaled = model.predict(X_test, verbose=0)
lstm_pred = scaler.inverse_transform(lstm_scaled).ravel()
lstm_mae = mean_absolute_error(y_test_actual, lstm_pred)
lstm_rmse = mean_squared_error(y_test_actual, lstm_pred) ** 0.5
pd.DataFrame({"model": ["Naive previous value", "LSTM"], "MAE": [naive_mae, lstm_mae], "RMSE": [naive_rmse, lstm_rmse]}).round(3)


## Plot actual vs predicted

A smooth line is not enough. Compare with the baseline and look for lagging behavior.


In [ ]:
test_days = test_df["day"].values[window_size:]
plt.figure(figsize=(10, 4))
plt.plot(test_days, y_test_actual, label="Actual", linewidth=2)
plt.plot(test_days, naive_pred, label="Naive baseline", alpha=0.8)
plt.plot(test_days, lstm_pred, label="LSTM prediction", alpha=0.8)
plt.xlabel("Day")
plt.ylabel("Value")
plt.title("Actual vs predicted values")
plt.legend()
plt.grid(alpha=0.3)
plt.show()


## Optional experiment: window size

Try `window_size = 7`, `30`, or `60`. Changing window size requires recreating windows and retraining the model.


## What to observe

- Time series validation must respect time order.
- Windowing converts a sequence into supervised learning samples.
- LSTM must be compared with a naive baseline.


## Common pitfalls

- Random split leaks future information.
- Scaling on full data leaks future information.
- LSTM input shape is easy to get wrong.
- Smooth predictions may simply follow recent values.
- LSTM is not automatically better than ARIMA or naive baseline.
- This is not a trading system.


## Try it yourself

- Change `window_size`.
- Increase or decrease LSTM units.
- Add another LSTM layer only if runtime remains reasonable.
- Change noise level in the synthetic data.
- Compare whether LSTM still beats the naive baseline.


## Reflection questions

- Why should time series data not be shuffled?
- What does each training sample represent after windowing?
- Did LSTM beat the naive baseline?
- If yes, by how much?
- If no, what could explain the result?


## Final summary

Time series validation must respect time. LSTM needs sliding windows. Baselines are essential. Visual quality alone is not enough evidence.
